# 🗃️ Conversão e Ajustes SQL no Banco de Dados de Crimes - MG

Este notebook documenta o processo de criação e ajuste do banco de dados SQLite com os dados de crimes violentos, bem como a integração com o banco de municípios do IBGE para padronização dos códigos municipais.

## 🧰 Scripts utilizados no processo

- `conversao_banco_sql_crimes.py`: Cria o banco SQLite principal `crimes_mg.sqlite` a partir do CSV unificado tratado `crimes_unificado_bruto.csv`.
- `conversao_ibge_sql.py`: Cria o banco SQLite `municipios_ibge.sqlite` a partir do arquivo `Municipios_pre_tratados.csv` (planilha tratada).
- `sql_normalizacao_codmunicipio.py`: Realiza as atualizações via SQL para normalizar o campo `codmunicipio` em `crimes_mg.sqlite`, usando os dados do banco IBGE, e padroniza os nomes dos municípios.


## 🧾 Sobre o banco de dados IBGE e tratamento dos 

Os dados do banco `municipios_ibge.sqlite` foram obtidos a partir das planilhas oficiais de estimativas populacionais e códigos dos municípios disponibilizados pelo IBGE:

- Fonte: [IBGE Estimativas de População 2024](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html?edicao=41105&t=downloads)
  
Essas planilhas foram tratadas manualmente via Excel para:

- Remover inconsistências e duplicações
- Padronizar o formato dos códigos municipais (`codcompleto` e `codincompleto`)
- Normalizar nomes dos municípios, incluindo remoção de acentos e caracteres especiais

Optamos por manter dois bancos SQLite separados (`crimes_mg.sqlite` e `municipios_ibge.sqlite`) para evitar a repetição dos dados de crimes em cada município, pois cada município possui múltiplos registros de crimes.

A integração entre os dois bancos é feita através da normalização dos códigos municipais via script SQL, garantindo consultas cruzadas consistentes.

Observações: Caso tenha interesse os arquivos csv estao disponibilizados dentro de docs.



## 🔄 1. Criando o banco SQLite principal com os dados de crimes

Aqui carregamos os dados do CSV tratado e criamos a tabela no banco `crimes_mg.sqlite`.

(Usamos o script `conversao_banco_sql_crimes.py` para essa tarefa.)

## 🔄 2. Criando o banco SQLite com os dados dos municípios do IBGE

A partir da planilha tratada `Municipios_pre_tratados.csv`, criamos o banco `municipios_ibge.sqlite` para armazenar os dados oficiais dos municípios.

(Usamos o script `conversao_ibge_sql.py`.)

## 🔄 3. Normalizando códigos e nomes via SQL

Realizamos as atualizações necessárias no banco `crimes_mg.sqlite` para:

- Atualizar o campo `codmunicipio` com o código completo do IBGE (`codcompleto`)
- Padronizar os nomes dos municípios em caixa alta (UPPER)
- Garantir a consistência entre os dados dos dois bancos para consultas futuras e uso no dashboard

(Usamos o script `sql_normalizacao_codmunicipio.py`.)

## 🧪 4. Testes e validações

Aqui fazemos consultas SQL para validar os ajustes e verificar que a normalização foi aplicada corretamente.

Exemplo:

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/database/crimes_mg.sqlite')

query = '''
SELECT municipio, codmunicipio, COUNT(*) as total_registros
FROM crimes
GROUP BY municipio, codmunicipio
ORDER BY total_registros DESC
LIMIT 10
'''

df = pd.read_sql_query(query, conn)
display(df)

conn.close()

,municipio,codmunicipio,total_registros
0,ABADIA DOS DOURADOS,3100104,2028
1,ABAETE,3100203,2028
2,ABRE CAMPO,3100302,2028
3,ACAIACA,3100401,2028
4,ACUCENA,3100500,2028
5,AGUA BOA,3100609,2028
6,AGUA COMPRIDA,3100708,2028
7,AGUANIL,3100807,2028
8,AGUAS FORMOSAS,3100906,2028
9,AGUAS VERMELHAS,3101003,2028


In [5]:
# Conectar ao banco principal de crimes
conn = sqlite3.connect('../data/database/crimes_mg.sqlite')
cursor = conn.cursor()

# Anexar o banco de municípios do IBGE para realizar join
cursor.execute("ATTACH DATABASE '../data/database/municipios_ibge.sqlite' AS ibge")

# Consulta SQL para calcular total de crimes e taxa percentual por município
query = """
SELECT 
    c.municipio,
    c.codmunicipio,
    SUM(c.registros) AS total_crimes,
    i.populacaoestimada,
    ROUND(CAST(SUM(c.registros) AS FLOAT) / i.populacaoestimada * 100, 4) AS taxa_percentual
FROM crimes c
JOIN ibge.municipios i ON c.codmunicipio = i.codcompleto
GROUP BY c.municipio, c.codmunicipio, i.populacaoestimada
ORDER BY taxa_percentual DESC
LIMIT 15
"""

# Executa a consulta e traz os dados para um DataFrame pandas
df = pd.read_sql_query(query, conn)

# Desanexar o banco IBGE para liberar recursos
cursor.execute("DETACH DATABASE ibge")
conn.close()

# Formata a coluna de taxa percentual para exibir como porcentagem com 4 casas decimais
df['taxa_percentual'] = df['taxa_percentual'].map(lambda x: f"{x:.4f}%")

# Exibe o DataFrame formatado
display(df)

,municipio,codmunicipio,total_crimes,populacaoestimada,taxa_percentual
0,CONTAGEM,3118601,104971,649975,16.1500%
1,BELO HORIZONTE,3106200,345115,2416339,14.2826%
2,BETIM,3106705,49985,429236,11.6451%
3,NOVA SERRANA,3145208,11526,112910,10.2081%
4,JUATUBA,3136652,3113,32726,9.5123%
5,SETE LAGOAS,3167202,20343,237931,8.5500%
6,NOVA PORTEIRINHA,3145059,575,6780,8.4808%
7,PERDIGAO,3149705,1087,12925,8.4101%
8,CONCEICAO DO PARA,3117603,453,5567,8.1372%
9,UBERABA,3170107,26988,354142,7.6207%


Observações:

A taxa percentual indica quantos crimes ocorreram para cada 100 habitantes, aproximado.

Essa métrica pode ajudar a identificar municípios com alta incidência relativa de crimes.

Pode ser usada como base para análises visuais, dashboards e monitoramento.